<a href="https://colab.research.google.com/github/Metropoliya/AI/blob/main/Book%20Voice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pymupdf torch torchaudio omegaconf

In [4]:
import fitz  # PyMuPDF
import torch
import re
import wave
import gc
import numpy as np
import os

# --- Настройки ---
PDF_PATH = "new 666.pdf"
BASE_OUTPUT_NAME = "hackers_of_dreams"
SPEAKER = 'xenia'
SAMPLE_RATE = 48000
CHUNKS_PER_FILE = 250  # Сколько фрагментов писать в один файл (250 - безопасно)

# ... (Шаги 1, 2 и 3 остаются без изменений) ...

# 4. Основной процесс генерации с автоматической разбивкой на файлы
text = get_clean_text_from_pdf(PDF_PATH)
chunks = chunk_text(text)

print(f"Текст разбит на {len(chunks)} фрагментов.")
print(f"Файлы будут разбиты по {CHUNKS_PER_FILE} фрагментов, чтобы не превысить лимит WAV.")

# Проходим по списку чанков с шагом CHUNKS_PER_FILE
for file_index in range(0, len(chunks), CHUNKS_PER_FILE):
    # Берем порцию чанков для текущего файла
    batch_chunks = chunks[file_index : file_index + CHUNKS_PER_FILE]

    # Формируем имя: hackers_of_dreams_part_1.wav, _part_2.wav и т.д.
    part_number = (file_index // CHUNKS_PER_FILE) + 1
    output_filename = f"{BASE_OUTPUT_NAME}_part_{part_number}.wav"

    print(f"\n--- Начинаем запись {output_filename} ({len(batch_chunks)} фрагментов) ---")

    with wave.open(output_filename, 'wb') as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(SAMPLE_RATE)

        for i, chunk in enumerate(batch_chunks):
            if not chunk.strip():
                continue

            try:
                # Генерация аудио
                audio = model.apply_tts(text=chunk, speaker=SPEAKER, sample_rate=SAMPLE_RATE)

                # Конвертация
                audio_np = (audio * 32767).to(torch.int16).cpu().numpy()
                wav_file.writeframes(audio_np.tobytes())

                # Пауза
                silence = torch.zeros(int(SAMPLE_RATE * 0.3), dtype=torch.int16).cpu().numpy()
                wav_file.writeframes(silence.tobytes())

                # Очистка памяти
                if (i + 1) % 10 == 0:
                    global_chunk_num = file_index + i + 1
                    print(f"Обработано всего {global_chunk_num} / {len(chunks)}...")
                    del audio
                    gc.collect()
                    if device.type == 'cuda':
                        torch.cuda.empty_cache()

            except Exception as e:
                print(f"Ошибка при обработке фрагмента {file_index + i}: {e}")

    print(f"Файл {output_filename} успешно сохранен!")

print("\n--- ВСЕ ЧАСТИ УСПЕШНО СГЕНЕРИРОВАНЫ! ---")

Извлекаем текст из PDF...
Текст разбит на 1253 фрагментов.
Файлы будут разбиты по 250 фрагментов, чтобы не превысить лимит WAV.

--- Начинаем запись hackers_of_dreams_part_1.wav (250 фрагментов) ---
Обработано всего 10 / 1253...
Обработано всего 20 / 1253...
Обработано всего 30 / 1253...
Обработано всего 40 / 1253...
Обработано всего 50 / 1253...
Обработано всего 60 / 1253...
Обработано всего 70 / 1253...
Обработано всего 80 / 1253...
Обработано всего 90 / 1253...
Обработано всего 100 / 1253...
Обработано всего 110 / 1253...
Обработано всего 120 / 1253...
Обработано всего 130 / 1253...
Обработано всего 140 / 1253...
Обработано всего 150 / 1253...
Обработано всего 160 / 1253...
Обработано всего 170 / 1253...
Обработано всего 180 / 1253...
Обработано всего 190 / 1253...
Обработано всего 200 / 1253...
Обработано всего 210 / 1253...
Обработано всего 220 / 1253...
Обработано всего 230 / 1253...
Обработано всего 240 / 1253...
Обработано всего 250 / 1253...
Файл hackers_of_dreams_part_1.wav у

In [5]:
import os

# 1. Создаем текстовый файл со списком всех наших частей по порядку
with open("file_list.txt", "w") as f:
    for i in range(1, 7): # От 1 до 6 включительно
        f.write(f"file 'hackers_of_dreams_part_{i}.wav'\n")

print("Список файлов создан. Начинаем склейку и конвертацию в MP3...")

# 2. Магия ffmpeg через консольную команду (знак ! позволяет запускать консоль в Colab)
# -f concat: режим склейки файлов из списка
# -c:a libmp3lame: кодировщик для MP3
# -q:a 2: высокое качество звука (примерно 190 kbps)
# -y: автоматически перезаписывать файл, если он уже существует
!ffmpeg -f concat -safe 0 -i file_list.txt -c:a libmp3lame -q:a 2 hackers_of_dreams_full_book.mp3 -y

print("\n--- ГОТОВО! Единый файл hackers_of_dreams_full_book.mp3 успешно создан! ---")

Список файлов создан. Начинаем склейку и конвертацию в MP3...
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame

In [10]:
import subprocess

file_path = "hackers_of_dreams_full_book.mp3"

try:
    # Запускаем системную утилиту ffprobe для чтения метаданных MP3
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries",
         "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", file_path],
        stdout=subprocess.PIPE,
        text=True
    )

    # Получаем длительность в секундах
    duration_seconds = float(result.stdout.strip())

    hours = int(duration_seconds // 3600)
    minutes = int((duration_seconds % 3600) // 60)

    print(f"Анализ файла: {file_path}")
    print(f"Длительность склеенного аудио: {hours} часов {minutes} минут")

except Exception as e:
    print(f"Ой, что-то пошло не так при чтении: {e}")

Анализ файла: hackers_of_dreams_full_book.mp3
Длительность склеенного аудио: 14 часов 32 минут


In [11]:
from google.colab import files

print("Начинаем сжатие идеального файла в MP3... Ждем пару минут.")

# Берем правильный файл (full.wav) и конвертируем его в MP3
!ffmpeg -i hackers_of_dreams_full.wav -c:a libmp3lame -q:a 2 hackers_of_dreams_perfect_book.mp3 -y

print("Готово! Начинаю скачивание...")

# Автоматически скачиваем готовый правильный MP3 на компьютер
files.download("hackers_of_dreams_perfect_book.mp3")

Начинаем сжатие идеального файла в MP3... Ждем пару минут.
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>